[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C15_Classic_Architectures_Course/00_setup/00_environment_check.ipynb)

# 00 · 环境自检与方法论热身

本课全程 **纯 numpy、CPU 可跑**，每个经典算子都从零实现，再与可信参考 **对拍**。

这个 notebook 做四件事：① 确认环境（numpy / scipy / sklearn）；② 加载本课要反复用的**真实数据**（optdigits 8×8 手写数字）；③ 立下全课纪律——**对拍 + 数值梯度检验**；④ 备好两个贯穿全课的小工具。

## 1 · 环境自检

必需 `numpy`；强烈建议 `scipy`（对拍卷积）、`scikit-learn`（真实数字数据）；`matplotlib` 可选。

In [ ]:
import sys, platform
print('Python', sys.version.split()[0], '|', platform.system())
import numpy as np
print('numpy', np.__version__)
for name in ['scipy', 'sklearn', 'matplotlib']:
    try:
        m = __import__(name)
        print(f'{name:12s}', getattr(m, '__version__', '?'))
    except Exception:
        print(f'{name:12s} 未安装', '(必需!)' if name=='scipy' else '(建议/可选)')
print('环境就绪 ✅')

## 2 · 真实数据：optdigits 8×8 手写数字

本课的图像实验用 `sklearn` 自带的 **optdigits**（1797 张 8×8 灰度手写数字，0–16 灰阶）。它足够小（CPU 秒级）又是**真实数据**，适合验证卷积/池化/CNN。

下面加载它，看一张数字的像素矩阵——后面模块 01/02 都会用它。

In [ ]:
from sklearn.datasets import load_digits
digits = load_digits()
X = digits.images          # (1797, 8, 8) 像素
y = digits.target          # (1797,) 标签 0..9
print('images:', X.shape, '| labels:', y.shape, '| 灰阶范围', X.min(), '~', X.max())
# 看第 0 张（数字 0）的像素矩阵
img0 = X[0]
print('第 0 张标签 =', y[0])
for row in img0.astype(int):
    print(' '.join(f'{v:2d}' for v in row))
assert X.shape == (1797, 8, 8) and X.max() <= 16
print('✅ optdigits 加载成功（本课的真实图像数据）')

## 3 · 立纪律之一：对拍（differential testing）

本课每个**有解析参考**的算子（卷积、池化）都要和一个可信参考比对，标准是 `np.allclose(mine, ref, atol=1e-10)`。

先用一个最小例子跑通工作流：手写一个朴素求和，对拍 `np.sum`。

In [ ]:
def my_sum(x):
    total = 0.0
    for v in x.ravel():
        total += float(v)
    return total

rng = np.random.default_rng(0)
x = rng.standard_normal(1000)
assert np.allclose(my_sum(x), x.sum(), atol=1e-9)
print('对拍通过：my_sum == np.sum ✅')
print('全课工作流：写算子 -> 对拍可信参考 -> assert 兜底。')

## 4 · 立纪律之二：数值梯度检验

反向传播（BPTT、LSTM、注意力的 backward）往往**没有现成参考**。这时用**有限差分**当 ground truth：

$$ \frac{\partial f}{\partial x_i} \approx \frac{f(x+\epsilon e_i) - f(x-\epsilon e_i)}{2\epsilon} $$

把它和解析梯度比，相对误差 < 1e-5 即认为反向写对了。这是模块 03/04/05 的金标准。

In [ ]:
def numerical_grad(f, x, eps=1e-5):
    '''中心差分逐元素估计 df/dx。f: ndarray->标量。'''
    g = np.zeros_like(x, dtype=float)
    it = np.nditer(x, flags=['multi_index'])
    while not it.finished:
        idx = it.multi_index
        old = x[idx]
        x[idx] = old + eps; fp = f(x)
        x[idx] = old - eps; fm = f(x)
        x[idx] = old
        g[idx] = (fp - fm) / (2 * eps)
        it.iternext()
    return g

def rel_error(a, b):
    return np.max(np.abs(a - b) / (np.maximum(1e-8, np.abs(a) + np.abs(b))))

# 演示：f(x)=sum(x^2)，解析梯度 2x
x = rng.standard_normal((3, 4))
f = lambda x: np.sum(x ** 2)
g_num = numerical_grad(f, x)
g_ana = 2 * x
err = rel_error(g_num, g_ana)
print(f'数值梯度 vs 解析梯度 相对误差 = {err:.2e}')
assert err < 1e-7, '数值梯度应与解析梯度吻合'
print('✅ 数值梯度检验工具就绪（模块 03/04/05 的反向都靠它验证）')

## 5 · 备好两个贯穿全课的小工具

数值稳定 softmax 与 one-hot 编码，后面卷积分类、注意力、seq2seq 都要用。

In [ ]:
def softmax(x, axis=-1):
    '''数值稳定 softmax：先减最大值再 exp。'''
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)

def one_hot(idx, n_classes):
    out = np.zeros((len(idx), n_classes))
    out[np.arange(len(idx)), idx] = 1.0
    return out

# 验证 softmax：每行和为 1、且不溢出
z = np.array([[1000.0, 1001.0, 1002.0]])    # 大 logit，朴素 exp 会溢出
s = softmax(z)
print('稳定 softmax:', np.round(s, 4), '| 行和 =', s.sum())
assert np.allclose(s.sum(axis=-1), 1.0) and not np.isnan(s).any()
# 验证 one-hot
oh = one_hot(np.array([0, 2, 1]), 3)
assert np.array_equal(oh, np.eye(3)[[0, 2, 1]])
print('✅ softmax / one_hot 就绪')

## 6 · 一个会贯穿全课的对拍工具

把「对拍」封装成统一裁判，后面每个模块都用它判定「我的算子 == 可信参考」。

In [ ]:
def check_allclose(name, got, ref, atol=1e-10):
    got = np.asarray(got, dtype=float); ref = np.asarray(ref, dtype=float)
    ok = np.allclose(got, ref, atol=atol)
    max_err = float(np.max(np.abs(got - ref))) if got.size else 0.0
    print(f'[{name:<28}] allclose={ok}  max|err|={max_err:.2e}')
    assert ok, f'{name} 与参考不一致！'
    return ok

check_allclose('demo: x vs x', x, x)
print('\n这就是全课的工作流：写算子 -> 对拍可信参考 / 数值梯度 -> assert 兜底。')

✅ 检查全部通过即环境就绪、方法论到位。

**本课的契约**：你写的每个前向都会对拍可信参考、每个反向都会过数值梯度检验；结构正确则数值一致，数值一致则逻辑可迁移到 PyTorch/TensorFlow。

**接下来六个模块**：01 卷积与池化 → 02 CNN 架构演化 → 03 RNN 与 BPTT → 04 LSTM 与 GRU → 05 seq2seq 与注意力 → 通向 Transformer（C1）。

下一站：**模块 01 · 卷积与池化**。